# Descarte Fácil — previsão de demanda de remoção de grandes objetos

**Pergunta:** é possível prever a quantidade de solicitações de remoção de grandes objetos por subprefeitura e mês, usando o histórico público do SP156?

**Fonte:** Portal de Dados Abertos da Prefeitura de São Paulo — Dados do SP156.
O conjunto é atualizado periodicamente e contém solicitações registradas pelos canais SP156.
Consulte a página oficial e o dicionário antes de cada execução:
https://dados.prefeitura.sp.gov.br/dataset/dados-do-sp156

> Importante: o SP156 registra solicitações, não todos os descartes irregulares existentes.
> Portanto, o modelo prevê demanda registrada, e não mede diretamente a quantidade real de lixo nas ruas.


In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn xgboost requests

In [ ]:
import io, requests, pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from xgboost import XGBRegressor

# CSV oficial mais recente identificado no portal (3º trimestre de 2026).
# Se a Prefeitura alterar o endereço, obtenha o novo link na página do conjunto.
DATA_URL = "https://dados.prefeitura.sp.gov.br/dataset/0aecfa2b-aa3a-40d4-8183-0d4351b7fd0a/resource/f67fcc12-8da4-485f-92a3-d2af40cc771e/download/arquivo-final-3-trim-26.csv"
r = requests.get(DATA_URL, timeout=180)
r.raise_for_status()
raw = pd.read_csv(io.BytesIO(r.content), sep=None, engine="python", encoding_errors="replace")
print(raw.shape)
display(raw.head())
print(raw.columns.tolist())

### Observação metodológica importante

A coluna `Data de Abertura` está no padrão **MM/DD/YYYY** (mês/dia/ano). Isso foi confirmado por valores como `07/25/2026`, em que `25` não pode representar um mês. A análise usa `Distrito` como unidade territorial, pois o CSV não contém uma coluna de Subprefeitura. A seleção de hiperparâmetros é feita na validação temporal; o conjunto de teste fica reservado para a avaliação final.


## 1. Preparação e seleção do serviço
Identifique as colunas de data, serviço/assunto e subprefeitura. Como os nomes podem variar entre extrações, confira `raw.columns` e ajuste o mapeamento abaixo conforme o dicionário oficial.

In [ ]:
# As colunas reais do CSV foram conferidas: "Data de Abertura", "Serviço" e "Distrito".
# O arquivo não possui uma coluna de Subprefeitura; a unidade territorial será Distrito.
date_col = "Data de Abertura"
service_col = "Serviço"
district_col = "Distrito"

missing = [c for c in [date_col, service_col, district_col] if c not in raw.columns]
if missing:
    raise ValueError(f"Colunas esperadas não encontradas: {missing}. Confira raw.columns.")

print("Data:", date_col, "| Serviço:", service_col, "| Unidade territorial:", district_col)


In [ ]:
df = raw.copy()

# O formato foi confirmado pelos próprios valores do CSV: por exemplo, 07/25/2026.
# Como 25 não pode ser mês, o padrão é mês/dia/ano (MM/DD/YYYY), não dia/mês/ano.
df[date_col] = pd.to_datetime(df[date_col].astype(str).str.strip(),
                              format="%m/%d/%Y", errors="coerce")
valid_dates = df[date_col].notna().mean()
print(f"Proporção de datas interpretadas: {valid_dates:.1%}")
if valid_dates < 0.95:
    raise ValueError("Menos de 95% das datas foram interpretadas. Verifique se o formato do CSV mudou.")

df = df.dropna(subset=[date_col, district_col, service_col]).copy()

# Seleção inicial por termos relacionados a remoção de grandes objetos.
print(df[service_col].astype(str).value_counts().head(40).to_string())


## 2. Agregação temporal e variável-alvo
Após conferir a nomenclatura exibida acima, ajuste `TERMS` para selecionar apenas o serviço de remoção de grandes objetos. A unidade de análise será subprefeitura-mês; o alvo é o número de solicitações naquele mês.

In [ ]:
TERMS = ["entulho e grandes objetos", "remoção de grandes objetos", "remocao de grandes objetos"]
service_text = df[service_col].astype(str).str.lower()
selected = df[service_text.apply(lambda x: any(t in x for t in TERMS))].copy()
if selected.empty:
    raise ValueError("Nenhum registro encontrado. Ajuste TERMS com o nome exato exibido em value_counts().")

selected["mes"] = selected[date_col].dt.to_period("M").dt.to_timestamp()
agg = (selected.groupby([district_col, "mes"]).size()
       .rename("solicitacoes").reset_index()
       .rename(columns={district_col:"regiao"}))
# Painel balanceado: meses sem solicitação observada recebem zero.
regions = agg["regiao"].dropna().unique()
months = pd.date_range(agg["mes"].min(), agg["mes"].max(), freq="MS")
panel = pd.MultiIndex.from_product([regions, months], names=["regiao","mes"]).to_frame(index=False)
panel = panel.merge(agg, on=["regiao","mes"], how="left")
panel["solicitacoes"] = panel["solicitacoes"].fillna(0).astype(int)
display(panel.head())
print("Período:", panel.mes.min(), "a", panel.mes.max(), "| linhas:", len(panel))

## 3. Engenharia de atributos e divisão treino/teste
Usamos calendário e histórico defasado. A divisão é cronológica para evitar que meses futuros contaminem o treinamento. O teste corresponde aos 20% finais dos meses disponíveis.

In [ ]:
panel = panel.sort_values(["regiao","mes"]).copy()
panel["ano"] = panel["mes"].dt.year
panel["mes_num"] = panel["mes"].dt.month
panel["trimestre"] = panel["mes"].dt.quarter

# Defasagens curtas: a série disponível é limitada a aproximadamente um ano.
for lag in [1, 2, 3]:
    panel[f"lag_{lag}"] = panel.groupby("regiao")["solicitacoes"].shift(lag)
panel["media_movel_2"] = panel.groupby("regiao")["solicitacoes"].transform(
    lambda s: s.shift(1).rolling(2).mean()
)

model_df = panel.dropna().copy()
X = pd.get_dummies(
    model_df[["regiao","ano","mes_num","trimestre","lag_1","lag_2","lag_3","media_movel_2"]],
    columns=["regiao"], drop_first=False, dtype=int
)
y = model_df["solicitacoes"]

# Separação temporal em treino, validação e teste. O teste fica isolado
# e não é utilizado para escolher hiperparâmetros.
months = np.array(sorted(model_df["mes"].unique()))
if len(months) < 6:
    raise ValueError("Poucos meses disponíveis após criar as defasagens para uma divisão temporal confiável.")

n_train = max(1, int(np.floor(len(months) * 0.60)))
n_val = max(1, int(np.floor(len(months) * 0.20)))
if n_train + n_val >= len(months):
    n_train = len(months) - 2
    n_val = 1

train_months = months[:n_train]
val_months = months[n_train:n_train+n_val]
test_months = months[n_train+n_val:]

train_mask = model_df["mes"].isin(train_months)
val_mask = model_df["mes"].isin(val_months)
test_mask = model_df["mes"].isin(test_months)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_trainval, y_trainval = X.loc[train_mask | val_mask], y.loc[train_mask | val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print("Treino:", train_months[0], "a", train_months[-1], "| linhas:", len(X_train))
print("Validação:", val_months[0], "a", val_months[-1], "| linhas:", len(X_val))
print("Teste:", test_months[0], "a", test_months[-1], "| linhas:", len(X_test))


## 4. Treinamento e comparação de hiperparâmetros
Comparamos uma linha de base (previsão igual ao mês anterior), Random Forest e XGBoost. Métricas: MAE, RMSE e R². O MAE é especialmente interpretável: erro médio em número de solicitações.

In [ ]:
# Hiperparâmetros são escolhidos APENAS na validação.
# Depois, cada modelo é reajustado com treino + validação e avaliado uma única vez no teste.
results, fitted = [], {}

baseline = model_df.loc[test_mask, "lag_1"].values
results.append({
    "modelo": "Baseline lag-1",
    "MAE": mean_absolute_error(y_test, baseline),
    "RMSE": mean_squared_error(y_test, baseline)**0.5,
    "R2": r2_score(y_test, baseline)
})

rf_grid = list(ParameterGrid({
    "n_estimators": [200, 400],
    "max_depth": [8, None],
    "min_samples_leaf": [1, 3]
}))
best_rf_params, best_rf_mae = None, float("inf")
for params in rf_grid:
    candidate = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    candidate.fit(X_train, y_train)
    val_pred = candidate.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_pred)
    if val_mae < best_rf_mae:
        best_rf_params, best_rf_mae = params, val_mae

rf = RandomForestRegressor(random_state=42, n_jobs=-1, **best_rf_params)
rf.fit(X_trainval, y_trainval)
fitted["Random Forest"] = rf
rf_pred = rf.predict(X_test)
results.append({
    "modelo": f"Random Forest {best_rf_params}",
    "MAE": mean_absolute_error(y_test, rf_pred),
    "RMSE": mean_squared_error(y_test, rf_pred)**0.5,
    "R2": r2_score(y_test, rf_pred)
})

xgb_grid = list(ParameterGrid({
    "n_estimators": [200, 400],
    "max_depth": [3, 5],
    "learning_rate": [0.03, 0.08]
}))
best_xgb_params, best_xgb_mae = None, float("inf")
for params in xgb_grid:
    candidate = XGBRegressor(
        random_state=42, objective="reg:squarederror", n_jobs=-1, **params
    )
    candidate.fit(X_train, y_train)
    val_pred = candidate.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_pred)
    if val_mae < best_xgb_mae:
        best_xgb_params, best_xgb_mae = params, val_mae

xgb = XGBRegressor(
    random_state=42, objective="reg:squarederror", n_jobs=-1, **best_xgb_params
)
xgb.fit(X_trainval, y_trainval)
fitted["XGBoost"] = xgb
xgb_pred = xgb.predict(X_test)
results.append({
    "modelo": f"XGBoost {best_xgb_params}",
    "MAE": mean_absolute_error(y_test, xgb_pred),
    "RMSE": mean_squared_error(y_test, xgb_pred)**0.5,
    "R2": r2_score(y_test, xgb_pred)
})

metrics = pd.DataFrame(results).sort_values("MAE")
display(metrics)


In [ ]:
best_name = metrics.loc[metrics["modelo"].str.startswith(("Random Forest", "XGBoost")), "modelo"].iloc[0]
best_model = fitted["Random Forest"] if best_name.startswith("Random Forest") else fitted["XGBoost"]
best_pred = best_model.predict(X_test)

plt.figure(figsize=(7,5))
plt.scatter(y_test, best_pred, alpha=.35)
lim = max(float(y_test.max()), float(np.max(best_pred)))
plt.plot([0,lim],[0,lim],"--")
plt.xlabel("Solicitações reais")
plt.ylabel("Solicitações previstas")
plt.title("Valores reais x previstos — melhor modelo")
plt.tight_layout()
plt.show()

if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values().tail(12)
    imp.plot(kind="barh", figsize=(8,5), title="Atributos mais importantes")
    plt.tight_layout()
    plt.show()


## 5. Interpretação para o cidadão

Preencha a conclusão com os resultados efetivamente obtidos após executar todas as células:
- Melhor modelo e hiperparâmetros: **[preencher a partir da tabela]**.
- MAE no teste: **[preencher]** solicitações por subprefeitura/mês.
- RMSE e R²: **[preencher]**.
- Atributos mais relevantes: **[preencher a partir do gráfico]**.

**Aplicação no Descarte Fácil:** a previsão pode apoiar a priorização de comunicação e planejamento de coleta em regiões com demanda historicamente elevada. O app pode usar a informação como indicador de contexto, sem afirmar que uma região terá descarte irregular com certeza.

**Limitações:** solicitações SP156 dependem de conhecimento e iniciativa do cidadão; não representam todos os descartes. O recorte por subprefeitura é menos preciso que o CEP/rua. Mudanças de nomenclatura, cobertura e políticas públicas podem afetar o padrão. Uma previsão agregada não determina o dia exato em que o caminhão passará.
